In [4]:
import sys, os
sys.path.append("../src")
import paths                                   # HF_HOME → D

from huggingface_hub import HfApi, hf_hub_download

REPO = "roneneldan/TinyStoriesInstruct"
files = HfApi().list_repo_files(REPO, repo_type="dataset")    # 저장소 안의 파일 목록
print("파일 목록:")
for f in files:
    print("  ", f)

# 이름에 valid가 들어간 txt 파일을 골라서 다운로드
val_name = next(f for f in files if "valid" in f.lower() and f.endswith(".txt"))
val_path = hf_hub_download(REPO, val_name, repo_type="dataset")   # D의 HF 캐시에 저장
print(f"\n받은 파일: {val_path} ({os.path.getsize(val_path) / 1e6:.1f} MB)\n")

with open(val_path, encoding="utf-8") as f:    # 인코딩을 직접 지정
    for i, line in enumerate(f):
        print(f"[{i:3d}] {line!r}")
        if i >= 40:
            break

파일 목록:
   .gitattributes
   TinyStories-Instruct-train.txt
   TinyStories-Instruct-valid.txt
   readme.md

받은 파일: D:\mini-llm-data\hf-cache\hub\datasets--roneneldan--TinyStoriesInstruct\snapshots\ee050ed1f8720795be342921335e821856a2b42e\TinyStories-Instruct-valid.txt (26.9 MB)

[  0] 'ooks. He said, "Yes, please. Read, please."\n'
[  1] 'His mom picked up the book and continued to read. Sam snuggled with his mom and listened. He forgot about the tower. He was happy.\n'
[  2] "Summary: Sam's tower of books falls down while his mom reads him a story about a dragon and a knight, but his mom comforts him and they continue to enjoy the story together.\n"
[  3] '<|endoftext|>\n'
[  4] 'Random sentence: They are very excited and want to fly too.\n'
[  5] 'Features: Dialogue\n'
[  6] 'Summary: Tom and Anna are excited to go on a holiday with their parents, and they fly on a big plane to a place with sun and sand.\n'
[  7] 'Story: \n'
[  8] 'Tom and Anna are brother and sister. They like to pla

In [5]:
import collections
import tiktoken

FIELDS = {"Words:": "words", "Features:": "features", "Summary:": "summary",
          "Random sentence:": "random_sentence", "Story:": "story"}
ORDER = ["words", "features", "summary", "random_sentence"]   # 템플릿에서의 조건 순서
LABEL = {"words": "Words", "features": "Features", "summary": "Summary",
         "random_sentence": "Random sentence"}


def parse_example(chunk):
    """예시 하나(여러 줄)를 필드별 dict로 분해"""
    ex, story_lines = {}, []
    for line in chunk.split("\n"):
        line = line.strip()
        if not line:
            continue
        for prefix, key in FIELDS.items():
            if line.startswith(prefix):
                value = line[len(prefix):].strip()
                if key == "story":
                    if value:                       # "Story:" 같은 줄에 본문이 시작되는 경우
                        story_lines.append(value)
                else:
                    ex[key] = value
                break
        else:                                       # 어떤 머리말로도 시작하지 않으면 본문
            story_lines.append(line)
    ex["story"] = "\n".join(story_lines)
    return ex


def build_prompt(ex):
    """조건 부분 (= 요청). Story: 까지 포함"""
    lines = [f"{LABEL[k]}: {ex[k]}" for k in ORDER if k in ex]
    return "\n".join(lines) + "\nStory:\n"


with open(val_path, encoding="utf-8") as f:
    chunks = f.read().split("<|endoftext|>\n")
chunks = chunks[1:]                                 # 첫 조각은 앞부분이 잘려 있음 → 버림
examples = [parse_example(c) for c in chunks]
examples = [e for e in examples if e["story"]]      # 본문 없는 조각 제거
print(f"예시 수: {len(examples):,}")

cnt = collections.Counter(k for e in examples for k in ORDER if k in e)
for k in ORDER:
    print(f"  {LABEL[k]:16s} {cnt[k] / len(examples):6.1%}")
print(f"  조건이 하나도 없음   {sum(not any(k in e for k in ORDER) for e in examples) / len(examples):6.1%}")

print("\n===== 템플릿 적용 예시 =====")
print(build_prompt(examples[0]) + examples[0]["story"][:300] + " ...")

# 토큰 길이: 우리 모델은 한 번에 256토큰까지만 봄
enc = tiktoken.get_encoding("gpt2")
p_len = [len(enc.encode(build_prompt(e))) for e in examples[:3000]]
s_len = [len(enc.encode(e["story"])) + 1 for e in examples[:3000]]   # +1: 끝의 EOT
total = [a + b for a, b in zip(p_len, s_len)]
print(f"\n프롬프트 평균 {sum(p_len) / len(p_len):.0f}토큰 / 이야기 평균 {sum(s_len) / len(s_len):.0f}토큰")
print(f"합계 256 초과 비율: {sum(t > 256 for t in total) / len(total):.1%}")

예시 수: 25,026
  Words             66.6%
  Features          54.2%
  Summary          100.0%
  Random sentence   30.2%
  조건이 하나도 없음     0.0%

===== 템플릿 적용 예시 =====
Features: Dialogue
Summary: Tom and Anna are excited to go on a holiday with their parents, and they fly on a big plane to a place with sun and sand.
Random sentence: They are very excited and want to fly too.
Story:
Tom and Anna are brother and sister. They like to play with their toys and read books. They are very happy because they are going on a holiday with their mum and dad. They will fly on a big plane to a place with a lot of sun and sand.
The day of the holiday comes and they pack their bags. They go to ...

프롬프트 평균 51토큰 / 이야기 평균 196토큰
합계 256 초과 비율: 29.9%


In [8]:
import random
import torch

random.seed(0)
BLOCK = 256


def drop_summary(ex, p=0.5):
    """Words나 Features가 있으면 p 확률로 Summary를 뺌 (condition dropout)"""
    if "summary" in ex and ("words" in ex or "features" in ex) and random.random() < p:
        ex = {k: v for k, v in ex.items() if k != "summary"}
    return ex


def encode_example(ex):
    """(x, y) 각 (256,) 반환. 256 초과면 None"""
    p_ids = enc.encode(build_prompt(ex))
    s_ids = enc.encode(ex["story"]) + [enc.eot_token]
    tokens = p_ids + s_ids                               # 길이 L
    if len(tokens) > BLOCK + 1:                          # x, y가 각각 256개가 되려면 L ≤ 257
        return None
    P = len(p_ids)
    pad = BLOCK + 1 - len(tokens)
    tokens = tokens + [enc.eot_token] * pad              # (257,) 뒤를 EOT로 채움
    x = torch.tensor(tokens[:-1])                        # (256,)
    y = torch.tensor(tokens[1:])                         # (256,) 한 칸 밀림
    y[:P - 1] = -100                                     # 정답이 프롬프트인 위치 무시
    if pad > 0:
        y[BLOCK - pad:] = -100                           # 패딩 위치 무시
    return x, y, P


for idx, e in enumerate(examples):
    ex = drop_summary(e)
    out = encode_example(ex)
    if out is not None:                  # 256 안에 들어가는 예시만
        break
x, y, P = out
print(f"examples[{idx}] 사용\n")
print(build_prompt(ex))
print(f"프롬프트 {P}토큰 / loss 계산 위치 {(y != -100).sum().item()}개 / 무시 위치 {(y == -100).sum().item()}개\n")

for i in list(range(P - 3, P + 3)) + [BLOCK - 1]:       # 프롬프트 → 이야기 경계 + 마지막 위치
    tgt = "  (무시)" if y[i] == -100 else repr(enc.decode([y[i].item()]))
    print(f"위치 {i:3d}  입력 {enc.decode([x[i].item()])!r:14s} → 정답 {tgt}")

kept = sum(encode_example(e) is not None for e in examples[:3000])
print(f"\n256에 들어가는 예시: {kept / 3000:.1%}")

examples[22] 사용

Words: lend, ghost, mighty
Features: Twist
Summary: Lily encounters a friendly ghost who turns into a superhero and rewards her for her kindness and bravery.
Story:

프롬프트 37토큰 / loss 계산 위치 169개 / 무시 위치 87개

위치  34  입력 'Story'        → 정답   (무시)
위치  35  입력 ':'            → 정답   (무시)
위치  36  입력 '\n'           → 정답 'Once'
위치  37  입력 'Once'         → 정답 ' upon'
위치  38  입력 ' upon'        → 정답 ' a'
위치  39  입력 ' a'           → 정답 ' time'
위치 255  입력 '<|endoftext|>' → 정답   (무시)

256에 들어가는 예시: 70.7%
